In [ ]:
import nltk
import os
from docling.document_converter import DocumentConverter
from nltk.tokenize import sent_tokenize

cache_path = os.path.abspath("./.cache/")


def get_text(path: str) -> str:
    converter = DocumentConverter()
    result = converter.convert(path)
    return result.document.export_to_text()


def get_sentences(text: str) -> list[str]:
    nltk.download("punkt_tab", download_dir=cache_path)
    if cache_path not in nltk.data.path:
        nltk.data.path.append(cache_path)
    return sent_tokenize(text)


def parse_triplet(decoded: str) -> list[dict]:
    triplets = []
    parts = decoded.replace("</s>", "").replace("__en__", "").split("__")
    data = {"sub": "", "rel": "", "obj": "", "sub_type": "", "obj_type": ""}
    for i in range(1, len(parts), 2):
        tag = parts[i]
        val = parts[i + 1].strip() if i + 1 < len(parts) else ""
        if tag == "sv":
            data["sub"] = val
        elif tag == "uk":
            data["obj"] = val
        elif tag == "wo":
            data["rel"] = val
        elif tag == "tn":
            data["sub_type"] = val
        elif tag == "yo":
            data["obj_type"] = val

        if data["sub"] and data["rel"] and data["obj"]:
            triplets.append(data)
            data = {"sub": data["sub"], "rel": "", "obj": "", "sub_type": "", "obj_type": ""}

    triplets.append(data)
    return triplets

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


def main():
    text = get_text("data/obama.pdf")
    text = text[text.find("\n\n") + 2:]  # remove title
    sentences = get_sentences(text)
    tokenizer = AutoTokenizer.from_pretrained("Babelscape/mrebel-base", src_lang="en", tgt_lang="en", cache_dir=cache_path)
    model = AutoModelForSeq2SeqLM.from_pretrained("Babelscape/mrebel-base", cache_dir=cache_path)
    lang_id = tokenizer.get_lang_id("en")
    for sentence in sentences:
        inputs = tokenizer(sentence, return_tensors="pt").to(model.device)
        generated_tokens = model.generate(**inputs, decoder_start_token_id=lang_id, max_length=256, num_beams=5)
        output = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)[0]
        print(f"Input: {sentence}")
        print(f"Output: {output}")
        print(f"Triplets: {parse_triplet(output)}")
        print()


main()